In [20]:
!pip install torchtext

In [21]:
from datasets import load_dataset
from google.colab import userdata
import pandas as pd
import torch
from sklearn.model_selection import train_test_split


In [22]:
import json

with open('/content/slmqa.json','r') as file:
  df = pd.read_json(file)

df

,question,answer
0,What is the name of the highest mountain in th...,everest
1,What is the name of the famous Austrian compos...,beethoven
2,Which country is the largest by area?,russia
3,"Who is the author of ""The Great Gatsby""?",fitzgerald
4,Who is credited with discovering penicillin?,fleming
...,...,...
902,What does the acronym CPU stand for?,central processing unit
903,Which mammal lays eggs?,platypus
904,What is the main alchoholic ingredient in a ma...,tequila
905,What were Jack and Jill going up the hill to f...,water


In [23]:
df.describe()
df['question']

,question
0,What is the name of the highest mountain in th...
1,What is the name of the famous Austrian compos...
2,Which country is the largest by area?
3,"Who is the author of ""The Great Gatsby""?"
4,Who is credited with discovering penicillin?
...,...
902,What does the acronym CPU stand for?
903,Which mammal lays eggs?
904,What is the main alchoholic ingredient in a ma...
905,What were Jack and Jill going up the hill to f...


In [24]:
x_temp , x_test,y_temp , y_test   = train_test_split(df['question'],df['answer'],test_size=0.2,random_state=42)
x_train,  x_val   , y_train ,y_val =train_test_split(x_temp,y_temp,test_size=0.2,random_state=42)
x_train = x_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
x_test = x_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [25]:
x_train.head()

,question
0,What is the term for a group of fish?
1,Who is the actress behind the character of Her...
2,What is the rarest blood type in humans?
3,What is the name of the smallest breed of dog?
4,What is the name of the second Harry Potter book?


In [26]:
y_train

,answer
0,school
1,emma watson
2,abnegative
3,chihuahua
4,chamber of secrets
...,...
575,lobster
576,jackson
577,christopher nolan
578,conductor


In [27]:
import nltk

nltk.download('punkt')
from nltk.tokenize import sent_tokenize ,word_tokenize


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [28]:
import nltk
from nltk.tokenize import word_tokenize
import pandas as pd

nltk.download('punkt_tab')
pd.set_option('display.max_colwidth', None)

corpus = df['question'].fillna(' ')+" "+df['answer'].fillna('')

def build_vocab(data):
  vocab = {}
  for sentence in data:
    for word in word_tokenize(sentence):
      vocab[word] = vocab.get(word, 0) + 1


  return vocab

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [29]:
vocab = build_vocab(corpus)

df_vocab = pd.DataFrame(vocab.items(), columns=['word', 'count'])
df_vocab.insert(loc=0,column='ID', value=range(1, len(df_vocab) + 1))
df_vocab

,ID,word,count
0,1,What,618
1,2,is,672
2,3,the,1230
3,4,name,209
4,5,of,587
...,...,...,...
2781,2782,going,1
2782,2783,hill,1
2783,2784,fetch,1
2784,2785,Afghanistan,1


In [30]:
len(vocab.keys())

2786

In [31]:
import sentencepiece as sp

In [32]:
with open('corpus.txt' , 'w', encoding='utf-8') as f:
  f.write('\n'.join(vocab.keys()))

In [33]:
sp.SentencePieceTrainer.train(
    input='corpus.txt',
    model_prefix='qa_spm',
    vocab_size=min(2455, len(vocab)),
    model_type="bpe",
    pad_id=0,                   # Explicitly define special tokens for PyTorch padding
    unk_id=1,
    bos_id=2,
    eos_id=3
)

True

In [34]:
tkz = sp.SentencePieceProcessor(model_file="qa_spm.model")

In [35]:
from torch.utils.data import Dataset, DataLoader

class TokenizeDataset(Dataset):
  def __init__(self, question , answer):
      self.question = question
      self.answer =answer
  def __len__(self):
      return len(self.question)
  def __getitem__(self, idx):
      item = (self.question[idx],self.answer[idx])
      return item

def collate_fn(batch):
  questions, answers = zip(*batch)
  ques_tokens = []
  ans_tokens = []
  answer_start_positions = []

  encode_ids = [ [tkz.bos_id()] + tkz.encode_as_ids(question) +[tkz.eos_id()] for question in questions]
  max_encode_id_len = max([len(encode_id) for encode_id in encode_ids])
  pad_encode_ids = [encode_id + [tkz.pad_id()]*(max_encode_id_len - len(encode_id)) for encode_id in encode_ids]
  encoder_input = torch.tensor(pad_encode_ids, dtype=torch.long)

  decode_ids = [ [tkz.bos_id()] + tkz.encode_as_ids(answer) +[tkz.eos_id()] for answer in answers] # Changed decode_as_ids to encode_as_ids
  max_decode_id_len = max([len(decode_id) for decode_id in decode_ids])
  pad_decode_ids = [decode_id + [tkz.pad_id()]*(max_decode_id_len - len(decode_id)) for decode_id in decode_ids]
  dec_tensor = torch.tensor(pad_decode_ids, dtype=torch.long)
  decoder_input = dec_tensor[:, :-1]
  target_ids = dec_tensor[:, 1:].clone()

  # mask pad positions so they don't contribute to loss
  target_ids[target_ids == tkz.pad_id()] = -100 # Changed sp.pad_id() to tkz.pad_id()

  return encoder_input, decoder_input, target_ids

In [36]:
train_dataset = TokenizeDataset(x_train, y_train)
val_dataset = TokenizeDataset(x_test, y_test)

In [39]:
dataset = TokenizeDataset(x_train, y_train)

print(type(x_train), len(x_train))
print(type(y_train), len(y_train))

print(dataset[0])
print(len(dataset))

<class 'pandas.core.series.Series'> 580
<class 'pandas.core.series.Series'> 580
__getitem__ returns: ('What is the term for a group of fish?', 'school'), type: <class 'tuple'>, len: 2
('What is the term for a group of fish?', 'school')
580


In [40]:
from torch.utils.data import DataLoader



train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [41]:
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, n_layers, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, n_layers, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, trg, hidden, cell):
        embedded = self.dropout(self.embedding(trg))
        outputs, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        predictions = self.fc_out(outputs)
        return predictions, hidden, cell



class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        hidden, cell = self.encoder(src)
        outputs, hidden, cell = self.decoder(trg, hidden, cell)
        return outputs





In [42]:
vocab_size = tkz.get_piece_size()

emb_dim = 128
hidden_dim = 256
n_layers = 1
dropout = 0

encoder = Encoder(vocab_size, emb_dim, hidden_dim, n_layers, dropout)
decoder = Decoder(vocab_size, emb_dim, hidden_dim, n_layers, dropout)
model = Seq2Seq(encoder, decoder)

In [43]:
criterion = nn.CrossEntropyLoss(ignore_index=-100)

In [44]:
import torch.optim as optim
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [47]:
num_epochs=150

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for encoder_input, decoder_input, target_ids in train_loader:
        optimizer.zero_grad()

        outputs = model(encoder_input, decoder_input)   # src, trg
        loss = criterion(
            outputs.reshape(-1, vocab_size),
            target_ids.reshape(-1)
        )

        loss.backward()
        optimizer.step()
        total_loss += loss.item()


    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Streaming output truncated to the last 5000 lines.
__getitem__ returns: ('Who is credited with saying "I think, therefore I am"?', 'descartes'), type: <class 'tuple'>, len: 2
__getitem__ returns: ('What term describes the study of the physical and chemical properties of sound waves?', 'acoustics'), type: <class 'tuple'>, len: 2
__getitem__ returns: ('What is the national animal of Australia?', 'kangaroo'), type: <class 'tuple'>, len: 2
__getitem__ returns: ('What is the primary language spoken in Brazil?', 'portuguese'), type: <class 'tuple'>, len: 2
__getitem__ returns: ('What color is the gemstone associated with the month of June?', 'pearl'), type: <class 'tuple'>, len: 2
__getitem__ returns: ('What is the fear of small spaces called?', 'claustrophobia'), type: <class 'tuple'>, len: 2
__getitem__ returns: ('What is a person who studies insects called?', 'entomologist'), type: <class 'tuple'>, len: 2
__getitem__ returns: ('What is the name for the contact point between two meeting su

In [ ]:
def generate_answer(question, max_len=20):
    model.eval()
    with torch.no_grad():
        q_ids = [tkz.bos_id()] + tkz.EncodeAsIds(question) + [tkz.eos_id()]
        encoder_input = torch.tensor([q_ids], dtype=torch.long)
        hidden, cell = encoder(encoder_input)

        decoder_input = torch.tensor([[tkz.bos_id()]], dtype=torch.long)
        generated = []

        for _ in range(max_len):
            logits, hidden, cell = decoder(decoder_input, hidden, cell)
            next_token = logits[:, -1, :].argmax(dim=-1).item()
            if next_token == tkz.eos_id():
                break
            generated.append(next_token)
            decoder_input = torch.tensor([[next_token]], dtype=torch.long)

    return tkz.DecodeIds(generated)

In [62]:
generate_answer("larger waterfall")

'victoria falls'

In [64]:
torch.save(model.state_dict(), 'qa_both_weights.pth')
